# Lab 3 · The pieces, under load — and how to test them

**Today:** when you walk out, the constructs from lab 2 work when combined — a loop inside a
function, an `if` chain inside a loop — and you can **write a test for a function someone else
wrote**, and say which tests are worth writing.

**Before you start:** lab 2 finished (or at least attempted). This lab fills the working hour.

Each section names one idea, explains what it does, and asks you to **predict
what a cell prints before you run it**. Write the prediction down — on paper,
out loud, in a comment. A wrong prediction you wrote down teaches you exactly
one thing; a shrug teaches you nothing.

Most sections end with a **Test your understanding** task: write a small piece
of code, then run the check cell under it. The check never grades and never
breaks anything — ⬜ means not attempted yet, ❌ means not yet (with a hint),
✅ means passing. The check cells are the one thing here to run rather than
edit; everything else is yours to break. From section 4 on, **you** write
checks of the same kind.

**AI in this lab:** until your prediction is written down, work at level 1 — no
AI. The prediction is how you find out what you, unaided, can already read, and
both exams are level 1. Once you have run a cell, level 3 is encouraged: ask
your tutor to explain any miss.

Labs are provided as aids to your understanding and are not graded. Feel free
to work together and consult AI if you get stuck. Run every cell. Change things.
Breaking this notebook costs nothing and teaches more than reading it.

## 1 · Read before you run

No new pieces today — combinations. **Predict what this prints — all of it, in order — before running.** The loop runs the chain once per value; exactly one branch fires each time.

In [ ]:
counts = [12, 0, 7, 30, 2]
small = 0
large = 0
for value in counts:
    if value < 5:
        small += 1
    elif value < 20:
        large += 0   # look carefully
    else:
        large += 1
print(small, large)

If you predicted `2 2` — look at the `elif` line again: it adds **zero**. The middle branch counts nothing, so values 12 and 7 fall through untallied. Reading code means reading what is written, not what is usual. That planted oddity is the kind both exams love.

**Test your understanding.** In the cell below, write the version that counts all three sizes properly: `n_small` (below 5), `n_medium` (5 up to but not including 20), `n_large` (20 and up), for the same list.

In [ ]:
# your turn: n_small, n_medium, n_large for counts = [12, 0, 7, 30, 2]
counts = [12, 0, 7, 30, 2]

In [ ]:
# run, don't edit — self-check
from labcheck import check

check("n_small", expect=2)
check("n_medium", expect=2, hint="5 up to but not including 20 — which values qualify?")
check("n_large", expect=1)

## 2 · A function of a function

Functions call functions; the reading skill is tracking what flows in and out. **Predict all three printed numbers.**

In [ ]:
def count_below(values, cutoff):
    count = 0
    for value in values:
        if value < cutoff:
            count += 1
    return count

def fraction_below(values, cutoff):
    return count_below(values, cutoff) / 4

draws = [0.1, 0.4, 0.6, 0.9]
print(count_below(draws, 0.5))
print(fraction_below(draws, 0.5))
print(fraction_below(draws, 1.0))

One thing to notice and one to distrust. Notice: `fraction_below` hands its inputs straight through to `count_below` — functions compose. Distrust: that hard-coded `/ 4` is only right for four-value lists — a bug lying in wait.

**Test your understanding.** Write `fraction_below_any(values, cutoff)` that works for a list of **any** length. You know a loop-based way to count how many values a list has (lab 2's `n_values` task) — or reuse `count_below` cleverly: every value is below a large enough cutoff.

In [ ]:
# your turn: fraction_below_any(values, cutoff) — correct for any list length

In [ ]:
# run, don't edit — self-check
from labcheck import check

check("fraction_below_any", expect=0.5, args=([0.1, 0.4, 0.6, 0.9], 0.5))
check("fraction_below_any", expect=0.2, args=([1, 2, 3, 4, 5], 1.5),
      hint="a five-value list — is your denominator still 4?")

## 3 · The generator, in a function

The habit that makes simulators testable: **the generator comes in as a parameter.** The function never creates its own — whoever calls decides the seed, so whoever calls can reproduce everything. Predict: will the two printed numbers be equal?

In [ ]:
import numpy as np

def count_heads(n_flips, chance, rng):
    heads = 0
    for _ in range(n_flips):
        if rng.random() < chance:
            heads += 1
    return heads

print(count_heads(50, 0.5, np.random.default_rng(21)))
print(count_heads(50, 0.5, np.random.default_rng(21)))

Equal — same seed, same fifty draws, same count. That is why every chunk in the textbook sets a seed, and why your problem-set answers can be checked at all.

**Test your understanding.** Write `longest_wait(chance, rng)`: flip until the first head (a draw below `chance`) and return how many flips it took, counting the head itself. Use `for attempt in range(1, 1001):` and `return attempt` the moment a head lands — `return` ends the function instantly, even mid-loop.

In [ ]:
# your turn: longest_wait(chance, rng) — flips until (and including) the first head
import numpy as np

In [ ]:
# run, don't edit — self-check
from labcheck import check

check("longest_wait", expect=1, args=(1.0, np.random.default_rng(3)),
      hint="at chance 1.0 the first flip is always a head")
check("longest_wait", expect=3, args=(0.5, np.random.default_rng(15)))

## 4 · A test is a claim you can check

You have been running check cells all week. Here is what is inside them.

The simplest test in Python is one line: `assert`, then a claim. If the claim is true, nothing
happens. If it is false, the cell stops with an `AssertionError`. That is the entire idea of a
software test — **a claim about what code returns, written down so a machine can check it** —
and every language has a version of it. Professional code ships with thousands.

Section 2's `count_below` is defined above. **Predict: does this cell print anything before its
last line?**

In [ ]:
assert count_below([0.1, 0.4, 0.6, 0.9], 0.5) == 2
assert count_below([], 0.5) == 0                  # an empty list: nothing to count
assert count_below([0.9, 0.9, 0.9], 0.5) == 0     # nothing below the cutoff
assert count_below([0.5, 0.5, 0.5], 0.5) == 0     # 0.5 is not below 0.5: < is strict
print("four claims, no complaints")

Nothing but the last line — every claim held. Now change the `2` in the first line to `3` and
re-run. The cell stops at that line with

```
AssertionError
```

and nothing after it runs. Read that as *the claim on this line is false*. Put the `2` back.

The check cells you have been running are the same idea with two changes: they **report instead
of stopping**, so one failed claim does not hide the rest, and each claim carries **a label in
words**. The function that does it is `check_that(label, condition)`, from the same `labcheck`
file the check cells use — ✅ with the label when the condition is `True`, ❌ when it is not.

**Test your understanding.** In the cell below, write **three** `check_that` claims about
`count_below`, each for a different input, and run the cell. Your own ✅ lines are the check
here — there is no check cell under this one. One to start from:

```python
check_that("nothing is below a cutoff of 0", count_below([0.1, 0.4, 0.6], 0.0) == 0)
```

In [ ]:
# your turn: three check_that claims about count_below, each on a different input
from labcheck import check_that


## 5 · A test can only catch what you thought of

Section 2's `fraction_below` has the bug you were told to distrust: it divides by 4, whatever
the list. Yet every test that comes to mind first —

```python
check_that("half of four", fraction_below([0.1, 0.4, 0.6, 0.9], 0.5) == 0.5)
```

— passes, because a four-value list is exactly the case the bug gets right. A test is worth
exactly as much as the input you chose for it.

**Test your understanding.** Build a list named `bug_input` on which `fraction_below` gives the
wrong answer, and write a `check_that` claim that says what the *right* answer is. Run the cell:
you should see ❌ — that is the test doing its job. Then point the same claim at your
`fraction_below_any` from section 2 and watch it pass. Leave the passing version in the cell.

In [ ]:
# your turn: bug_input — a list fraction_below gets wrong — and a check_that claim about it
from labcheck import check_that

In [ ]:
# run, don't edit — self-check
from labcheck import ready, check_that

if ready("bug_input", "fraction_below_any"):
    n = 0
    for _ in bug_input:
        n += 1
    check_that("bug_input has some length other than 4", n != 4,
               detail=f"it has {n} values — the bug is invisible on four-value lists")
    check_that("fraction_below gets bug_input wrong",
               fraction_below(bug_input, 0.5) != fraction_below_any(bug_input, 0.5),
               detail="on this list the two functions agree — the / 4 has to be wrong for it")

## 6 · Which tests are worth writing: forced inputs, then relationships

Section 3's `count_heads(n_flips, chance, rng)` has randomness in it, so you cannot say what it
returns for 50 flips at chance 0.5. But some inputs **force** the answer without a single draw
being seen — you met three in lab 2's last section. Those are the first tests to write for any
simulator: chance 0 (nothing lands), chance 1 (everything lands), zero flips (the loop never
runs).

Forced inputs catch a lot. What they miss is everything in between — code can be right at both
ends and nonsense in the middle. The test for the middle is a **relationship that must hold**:
more chance, more heads. One run cannot show it (50 flips at 0.5 might beat 50 flips at 0.6 by
luck), so you run many and compare the averages. **Predict which of the two printed averages is
larger, then run.**

In [ ]:
import numpy as np

rng = np.random.default_rng(4)
results = []
for _ in range(200):
    results.append(count_heads(50, 0.3, rng))
total = 0
for value in results:
    total += value
average = total / 200
print(f"average heads at chance 0.3, over 200 runs: {average:.2f}")

results = []
for _ in range(200):
    results.append(count_heads(50, 0.7, rng))
total = 0
for value in results:
    total += value
average = total / 200
print(f"average heads at chance 0.7, over 200 runs: {average:.2f}")

Two things to read in that cell. The shape — **append into a list inside a loop, then take one
number from the list after it** — is the pattern under every histogram in chapter 1 and under
most of chapter 2, where you will repeat an experiment a thousand times and ask what usually
happens. And the `print` line uses an **f-string**: `f"...{average:.2f}"` glues the value of
`average` into the text, rounded to two decimals for display only. You do not need to write
f-strings this term; you do need to read them.

**Test your understanding.** Write `average_heads(n_flips, chance, n_runs, rng)`: run
`count_heads(n_flips, chance, rng)` `n_runs` times, collect the results, and return their
average. Then, in the same cell, write **four** `check_that` claims about it — three forced
(chance 0, chance 1, and one more of your choosing) and one relationship (a higher chance gives
a higher average, over 200 runs each). Run the check cell only after your own four pass.

In [ ]:
# your turn: average_heads(n_flips, chance, n_runs, rng), then four check_that claims about it
import numpy as np
from labcheck import check_that

In [ ]:
# run, don't edit — self-check
import numpy as np
from labcheck import check, ready, check_that

check("average_heads", expect=50.0, args=(50, 1.0, 5, np.random.default_rng(1)),
      hint="every flip lands at chance 1.0, so every run is 50 and so is the average")
check("average_heads", expect=0.0, args=(50, 0.0, 5, np.random.default_rng(1)))
check("average_heads", expect=0.0, args=(0, 0.5, 5, np.random.default_rng(1)),
      hint="zero flips per run: what does count_heads return, and what is the average of five of those?")
if ready("average_heads"):
    low = average_heads(50, 0.3, 200, np.random.default_rng(2))
    high = average_heads(50, 0.7, 200, np.random.default_rng(2))
    check_that("a higher chance gives a higher average", high > low,
               detail=f"0.3 gave {low}, 0.7 gave {high}")

## 7 · How often? — the question chapter 2 asks

Once you have a list of results, the natural question is not the average but **how often**: in
what fraction of the runs did the count reach some value? Chapter 1 asked it with a tolerance —
how often does a run land near the observed count — and chapter 2 will ask it about extremes.
Same loop either way: go through the results, count the ones that pass the test, divide by how
many there were.

**Test your understanding.** Write `fraction_at_least(results, target)`: the fraction of values
in `results` that are **at or above** `target`. Count how many values there are with a loop, as
in lab 2 — do not hard-code the denominator. Then use it on 200 runs of `count_heads(50, 0.5,
rng)` and ask how often a fair coin gives **30 or more heads in 50 flips**. Write the number
down; you will see its like again on Friday.

In [ ]:
# your turn: fraction_at_least(results, target) — the share of values at or above target
import numpy as np

In [ ]:
# run, don't edit — self-check
from labcheck import check

check("fraction_at_least", expect=0.5, args=([38, 45, 40, 30], 40))
check("fraction_at_least", expect=1.0, args=([38, 45, 40, 30], 0))
check("fraction_at_least", expect=0.0, args=([38, 45, 40, 30], 46),
      hint="45 is below 46 — nothing reaches the target, so the fraction is 0.0")
check("fraction_at_least", expect=0.2, args=([1, 2, 3, 4, 5], 5),
      hint="a five-value list — is your denominator still 4?")

## If you finish early

- Take the relationship test from section 6 and make it fail on purpose: change `<` to `>` in
  `count_heads` and re-run your four claims. Which of them catch it? Which forced inputs pass
  anyway, and why? Put the `<` back.
- In section 7, ask for 35 or more heads instead of 30, then 40. Predict the direction the
  fraction moves before each run.
- Chapter 1's practice problems 1.1 and 1.2 are the same skills at full strength, and Friday's
  reading — chapter 2 — is where the loop in section 6 becomes the whole method.

## If you are stuck

Wave someone over — this hour exists so a stuck step costs you a minute rather
than an evening. Known snags:

- **The check for `longest_wait` never passes** — is your `range` starting at 1? Starting at 0
  makes every answer one too small.
- **`fraction_below_any` returns 0** — if you counted with `n_values`, check it is not still 0
  when you divide (a loop that never ran).
- **`NameError: name 'check_that' is not defined`** — the import line is the first line of the
  cell; if you deleted it, put `from labcheck import check_that` back.
- **Your section 5 claim shows ✅ on the first try** — then `bug_input` has four values, and the
  bug cannot be seen there. Any other length will do.
- A check stays ❌ and the hint has stopped helping — that is exactly what the room is for.